# Ejercicio 2: Implementación de la Distancia de Gower

El script debe contener una implementación de la Distancia de Gower:

1. Implementar la fórmula de Gower.
2. Validar con un subconjunto pequeño (ej. 5 filas).
3. Probar el cálculo de la matriz de distancias en una muestra pequeña del dataset.

In [ ]:
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo
from IPython.display import display

# =================================================================
# Carga del DataFrame Original (con valores faltantes)
# =================================================================
# Obtener el dataset "Adult" (id=2)
adult = fetch_ucirepo(id=2)

# Data como DataFrames de pandas
X = adult.data.features
y = adult.data.targets
df = pd.concat([X, y], axis=1)

# Reemplazar el caracter '?' por NaN de NumPy para un manejo estándar
df.replace('?', np.nan, inplace=True)

# =================================================================
# Implementación de la Distancia de Gower con Variables Ordinales
# =================================================================

# 1. Preparación de los datos para Gower
# -----------------------------------------------------------------
# Tomamos una muestra de 5 filas.
sample_indices = [27, 30, 31, 32, 33]
df_sample = df.loc[sample_indices].copy()

# Identificar tipos de variables
numeric_cols = ['age', 'fnlwgt', 'capital-gain', 'capital-loss', 'hours-per-week', 'education-num']
categorical_cols = ['workclass', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country', 'income']
ordinal_cols = ['education']

# --- Manejo de la Variable Ordinal 'education' ---

# a. Crear un mapa con el orden jerárquico de la variable
education_map = {
    'Preschool': 1, '1st-4th': 2, '5th-6th': 3, '7th-8th': 4, '9th': 5,
    '10th': 6, '11th': 7, '12th': 8, 'HS-grad': 9, 'Some-college': 10,
    'Assoc-voc': 11, 'Assoc-acdm': 12, 'Bachelors': 13, 'Masters': 14,
    'Prof-school': 15, 'Doctorate': 16
}

# b. Convertir la variable ordinal a rangos numéricos en una nueva columna
df['education_rank'] = df['education'].map(education_map)

# c. Calcular el rango (máx - mín) de los ranks ordinales del dataset completo
min_rank = df['education_rank'].min()
max_rank = df['education_rank'].max()
ordinal_rank_range = max_rank - min_rank

# d. Normalizar la variable ordinal en nuestra muestra a una escala [0, 1]
# Reemplazamos la columna 'education' en la muestra por su valor normalizado
df_sample['education'] = (df_sample['education'].map(education_map) - min_rank) / ordinal_rank_range


# Pre-calcular los rangos para las variables numéricas
ranges = {col: df[col].max() - df[col].min() for col in numeric_cols}


# 2. Definición de la función de Distancia de Gower (actualizada)
# -----------------------------------------------------------------
def gower_distance_missing(row1, row2):
    """
    Calcula la distancia de Gower, manejando variables numéricas, categóricas y ordinales.
    """
    total_dissimilarity = 0
    weight_sum = 0 # Denominador de la fórmula: Suma de los pesos w_ijk

    # Iterar sobre todas las columnas para calcular la disimilitud
    for col in df_sample.columns:
        # Ignoramos la columna auxiliar que creamos en el df completo
        if col == 'education_rank':
            continue

        val1, val2 = row1[col], row2[col]

        if pd.isna(val1) or pd.isna(val2):
            continue

        weight_sum += 1

        # a. Variable Numérica
        if col in numeric_cols:
            if ranges[col] > 0:
                total_dissimilarity += abs(val1 - val2) / ranges[col]

        # b. Variable Ordinal (pre-procesada)
        elif col in ordinal_cols:
            # Ya está normalizada en [0, 1], solo calculamos la diferencia absoluta
            total_dissimilarity += abs(val1 - val2)

        # c. Variable Categórica
        elif col in categorical_cols:
            if val1 != val2:
                total_dissimilarity += 1

    if weight_sum == 0:
        return 1.0

    return total_dissimilarity / weight_sum

# 3. Validación: Calcular la matriz de distancias para la muestra
# -----------------------------------------------------------------
n_samples = len(df_sample)
distance_matrix = np.zeros((n_samples, n_samples))

for i in range(n_samples):
    for j in range(n_samples):
        row_i = df_sample.iloc[i]
        row_j = df_sample.iloc[j]
        distance_matrix[i, j] = gower_distance_missing(row_i, row_j)

# Convertir a DataFrame para una visualización más clara
gower_matrix_df = pd.DataFrame(distance_matrix,
                               index=df_sample.index,
                               columns=df_sample.index)

print("\n--- Validación de la Distancia de Gower con Variables Ordinales ---")
print("Matriz de Distancias de Gower resultante:")
display(gower_matrix_df)